# 05 — Synthetic Control

Goal: 'Southeast rollout case study — what would the Southeast's failure rate have been if we'd never made the design change?'

Synthetic control constructs a weighted combination of control regions that matches the
Southeast's PRE-treatment trajectory. Post-treatment gap = causal effect.

Strength: works even when parallel trends doesn't hold, as long as the pre-treatment fit is good.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/field_panel.csv', parse_dates=['install_date'])
df['install_month_num'] = df.install_date.dt.month + (df.install_date.dt.year - 2022) * 12

# Monthly failure rate by region
monthly = df.groupby(['install_month_num', 'region']).failure_event.mean().reset_index()
panel = monthly.pivot(index='install_month_num', columns='region', values='failure_event') * 100
print(panel.head())

%matplotlib inline

In [ ]:
# Try pysyncon for synthetic control
try:
    from pysyncon import Dataprep, Synth
    print('pysyncon available — using for synthetic control')
    USE_PYSYNCON = True
except ImportError:
    print('pysyncon not installed — using hand-implemented weighted average')
    USE_PYSYNCON = False

In [ ]:
# Hand-implemented synthetic control (OLS weights on pre-period)
TREATED_REGION = 'Southeast'
CONTROL_REGIONS = [c for c in panel.columns if c != TREATED_REGION]
TREATMENT_START = 13  # month 13 = Jan 2023

pre_data   = panel[panel.index < TREATMENT_START]
post_data  = panel[panel.index >= TREATMENT_START]

# Solve for weights: minimize ||Southeast_pre - sum(w_i * control_i_pre)||
from scipy.optimize import minimize

X_pre = pre_data[CONTROL_REGIONS].values  # shape: (n_pre, n_controls)
y_pre = pre_data[TREATED_REGION].values

def objective(w):
    synth = X_pre @ w
    return np.sum((y_pre - synth) ** 2)

def constraint_sum_one(w): return np.sum(w) - 1.0

n_controls = len(CONTROL_REGIONS)
result = minimize(
    objective, x0=np.ones(n_controls) / n_controls,
    constraints={'type': 'eq', 'fun': constraint_sum_one},
    bounds=[(0, 1)] * n_controls,
)

weights = result.x
print('Synthetic control weights:')
for r, w in zip(CONTROL_REGIONS, weights): print(f'  {r}: {w:.3f}')

In [ ]:
# Construct synthetic control trajectory
synth_pre  = pre_data[CONTROL_REGIONS].values @ weights
synth_post = post_data[CONTROL_REGIONS].values @ weights

synth_full = np.concatenate([synth_pre, synth_post])
actual_full = panel[TREATED_REGION].values

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
t = panel.index
ax.plot(t, actual_full,   label='Southeast (actual)', color='steelblue', linewidth=2)
ax.plot(t, synth_full,    label='Synthetic Southeast', color='tomato', linestyle='--', linewidth=2)
ax.axvline(TREATMENT_START, color='gold', linestyle='--', label='Treatment start')
ax.fill_between(t[t >= TREATMENT_START],
                actual_full[t >= TREATMENT_START],
                synth_post, alpha=0.2, color='steelblue', label='Estimated effect')
ax.set_title('Synthetic Control — Southeast vs. Synthetic Counterfactual')
ax.set_xlabel('Install month'); ax.set_ylabel('Monthly failure rate (%)')
ax.legend()
plt.savefig('../figures/synthetic_control.png', dpi=150, bbox_inches='tight')
plt.show()

gap_post = actual_full[t >= TREATMENT_START] - synth_post
print(f'Average post-treatment gap: {gap_post.mean():+.2f}% (negative = variant B reduced failures)')